<a href="https://colab.research.google.com/github/ektam9931/flyrank-ml-week1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ektam9931/flyrank-ml-week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose Random Forest Regression because this task predicts future content performance, which is a regression problem. Random Forest can capture nonlinear relationships between SEO, traffic, and engagement signals while also allowing feature importance analysis. I will compare it against my Week-4 baseline using the same evaluation data and metric.

In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/ektam9931/flyrank-ml-week1/main/data/raw/content_refresh_anonymized.csv"
)

In [4]:
df.shape

(30000, 44)

In [5]:
 df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [6]:
 df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [7]:
df[['content_id', 'client_id', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d']].head()

,content_id,client_id,clicks_90d,clicks_last_30d,clicks_prev_30d
0,content_304f48230142,client_f369cb89fc,29,2,13
1,content_a1fb4e703a9e,client_4e07408562,7,2,1
2,content_9aa793d4d895,client_7f2253d7e2,11,1,3
3,content_331d6c4de07b,client_19581e27de,58,22,17
4,content_d99b7a2d90ca,client_3fdba35f04,24,10,2


In [8]:
df.head(2).T

,0,1
content_id,content_304f48230142,content_a1fb4e703a9e
client_id,client_f369cb89fc,client_4e07408562
search_volume,10.0,90.0
competition,0.67,0.01
competition_level,HIGH,LOW
cpc,2.05,0.05
content_type,keyword article,keyword article
main_intent,transactional,informational
word_count,3221.0,2481.0
char_count,20457.0,15562.0


In [9]:
df[["content_id", "client_id", "clicks_90d", "clicks_last_30d", "clicks_prev_30d"]].describe()

,clicks_90d,clicks_last_30d,clicks_prev_30d
count,30000.000000,30000.000000,30000.000000
mean,16.097333,4.933867,5.435100
std,75.076958,23.929393,28.358673
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000
75%,7.000000,2.000000,2.000000
max,4178.000000,1176.000000,1627.000000


In [11]:
 df[["clicks_90d", "clicks_last_30d", "clicks_prev_30d"]].head()

,clicks_90d,clicks_last_30d,clicks_prev_30d
0,29,2,13
1,7,2,1
2,11,1,3
3,58,22,17
4,24,10,2


In [13]:
df["click_change"].describe()

,click_change
count,30000.000000
mean,-0.501233
std,11.170325
min,-678.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,528.000000


In [17]:
df["click_change"] = df["clicks_last_30d"] - df["clicks_prev_30d"]

In [18]:
df["click_change"].value_counts().head(10)

,count
click_change,
0,17160
-1,2813
1,2708
-2,1122
2,1015
-3,636
3,581
-4,407
4,391


In [19]:
target = "click_change"

In [20]:
df[["click_change", "clicks_last_30d", "clicks_prev_30d"]].corr()

,click_change,clicks_last_30d,clicks_prev_30d
click_change,1.000000,-0.199818,-0.562503
clicks_last_30d,-0.199818,1.000000,0.922519
clicks_prev_30d,-0.562503,0.922519,1.000000


In [22]:
drop_cols = [
    "content_id",
    "client_id",
    "click_change",
    "clicks_90d",
    "clicks_last_30d",
    "clicks_prev_30d"
]

X = df.drop(columns=drop_cols)
y = df["click_change"]

In [23]:
X.shape, y.shape

((30000, 39), (30000,))

In [24]:
X.dtypes.value_counts()

,count
int64,16
object,12
float64,11


In [26]:

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

In [27]:
print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']
Numeric: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']


In [28]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [29]:
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

Train: (23837, 39) (23837,)
Test: (6163, 39) (6163,)
Train clients: 25
Test clients: 7


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [31]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [32]:
target = "click_change"

In [33]:
drop_cols = [
    "content_id",
    "client_id",
    "click_change",
    "clicks_90d",
    "clicks_last_30d",
    "clicks_prev_30d"
]

X = df.drop(columns=drop_cols)
y = df["click_change"]

## 2. Split design
I use a grouped train-test split by client_id. The dataset does not contain an observation date that would support a chronological split, so I do not claim this is time-aware. Instead, entire clients are held out from training and used only for evaluation. This tests whether the model generalizes to unseen clients and prevents observations from the same client appearing in both train and test.

In [34]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


In [35]:
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

Train: (23837, 39) (23837,)
Test: (6163, 39) (6163,)
Train clients: 25
Test clients: 7


## 3. Train + compare vs my baseline

I will train a Random Forest Regression model and compare it with a simple zero-change baseline. The target is the observed change in clicks between the last 30-day period and the previous 30-day period. Both approaches will be evaluated on the same held-out client groups using MAE, RMSE, and R². The comparison will show whether the model adds predictive value beyond the simple assumption of no change.

In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [38]:
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scroll_events_90d',
                                                   'days_with_impressions',
                                                   'days_w...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier',
                                                   'trend_direction'])])),
                ('model',
                 RandomForestRegressor(max_depth=12, n_estimators=200,
                                       n_jobs=-1, random_state=42))])

In [39]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest MAE:", mae)
print("Random Forest RMSE:", rmse)
print("Random Forest R²:", r2)

Random Forest MAE: 1.459439765519641
Random Forest RMSE: 6.776036971129557
Random Forest R²: 0.5644267109409592


In [41]:
baseline_pred = np.zeros(len(y_test))

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 1.897939315268538
Baseline RMSE: 10.267309956364503
Baseline R²: -5.307954860112041e-05


In [42]:
comparison = pd.DataFrame({
    "Model": ["Zero-change baseline", "Random Forest"],
    "MAE": [baseline_mae, mae],
    "RMSE": [baseline_rmse, rmse],
    "R2": [baseline_r2, r2]
})

comparison

,Model,MAE,RMSE,R2
0,Zero-change baseline,1.897939,10.267310,-0.000053
1,Random Forest,1.459440,6.776037,0.564427


## 4. Errors and interpretation

The model's largest errors occur on pages with unusually large changes in clicks. This suggests that the model has difficulty predicting extreme performance changes. The feature-importance results show which historical SEO, traffic, and engagement signals the model relied on most. Because the target is concentrated around zero, the model may perform well on typical pages while struggling with large positive or negative changes. These results make the model useful as decision-support rather than as a guarantee of future performance.

In [43]:
errors = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred
})

errors["error"] = errors["actual"] - errors["predicted"]
errors["absolute_error"] = errors["error"].abs()

errors.sort_values(
    "absolute_error",
    ascending=False
).head(10)


,actual,predicted,error,absolute_error
3898,448,80.510995,367.489005,367.489005
5429,-209,-87.850461,-121.149539,121.149539
2413,-129,-19.306916,-109.693084,109.693084
4062,-83,-159.352590,76.352590,76.352590
506,-227,-159.320026,-67.679974,67.679974
5823,-156,-89.290537,-66.709463,66.709463
4395,-84,-18.923053,-65.076947,65.076947
4835,86,20.997128,65.002872,65.002872
5547,88,24.506668,63.493332,63.493332
5544,-67,-127.101883,60.101883,60.101883


In [44]:
print("Mean absolute error:", errors["absolute_error"].mean())
print("Largest absolute error:", errors["absolute_error"].max())

print("\nLargest overpredictions:")
print(
    errors.sort_values("error", ascending=False).head(5)
)

print("\nLargest underpredictions:")
print(
    errors.sort_values("error", ascending=True).head(5)
)

Mean absolute error: 1.459439765519641
Largest absolute error: 367.4890046305046

Largest overpredictions:
      actual   predicted       error  absolute_error
3898     448   80.510995  367.489005      367.489005
4062     -83 -159.352590   76.352590       76.352590
4835      86   20.997128   65.002872       65.002872
5547      88   24.506668   63.493332       63.493332
5544     -67 -127.101883   60.101883       60.101883

Largest underpredictions:
      actual   predicted       error  absolute_error
5429    -209  -87.850461 -121.149539      121.149539
2413    -129  -19.306916 -109.693084      109.693084
506     -227 -159.320026  -67.679974       67.679974
5823    -156  -89.290537  -66.709463       66.709463
4395     -84  -18.923053  -65.076947       65.076947


In [45]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

importances = model.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

feature_importance.head(15)

,feature,importance
15,num__sessions_last_30d,0.193666
17,num__sessions_prev_30d,0.174987
11,num__scroll_events_90d,0.135519
26,num__trend_pct,0.104338
9,num__engaged_sessions_90d,0.096973
16,num__impressions_prev_30d,0.029276
21,num__ctr,0.027812
22,num__avg_position,0.025230
24,num__scroll_rate,0.022629
6,num__pageviews_90d,0.017620


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- The task is framed as a regression problem predicting `click_change`.
- Random Forest Regression was chosen because it can model nonlinear relationships between SEO, traffic, and engagement signals.
- The train-test split was performed by `client_id`, keeping entire clients separate between training and testing.
- The split is not described as time-aware because the dataset does not contain an observation date.
- The Random Forest was evaluated using MAE, RMSE, and R².
- A zero-change numerical baseline was created on the same test set for a valid comparison.
- Error analysis was performed to inspect the largest prediction errors.
- Feature importance was examined to understand which transformed features contributed most to the model.
- The model is treated as decision-support rather than a guarantee of future performance.